In [18]:
from peint.models.modules.ctmc_module import CTMCModule
from scripts.vep import CosineDMSAnalyzer, ThriftyDMSAnalyzer
from scripts.vep.utils import nt_wt_seqs
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import pandas as pd

# Load CoSiNE

In [19]:
CKPT_DIR = Path("/scratch/users/stephen.lu/projects/protevo/logs/train/runs") # replace with local path to checkpoint directory
cosine_ckpt_path = CKPT_DIR / "2026-01-06_18-32-49/checkpoints/epoch_001.ckpt"

cosine_model = CTMCModule.load_from_checkpoint(cosine_ckpt_path)
cosine_model = cosine_model.eval()

/scratch/users/aakarshv/venvs/peint_env/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'net' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['net'])`.


# Expression VEP

In [20]:
EXPRESSION_DIR = Path("scripts/vep/data_dms/expression")
t_values = [0.2]

# Koenig (heavy and light)
koenig_expr_path = EXPRESSION_DIR / "Koenig2017_g6_er.csv"
koenig_expr_analyzer_cosine = CosineDMSAnalyzer(koenig_expr_path, cosine_model)
koenig_expr_results_cosine = koenig_expr_analyzer_cosine.compute_results(t_values, chains=["heavy", "light"], paired=True)

koenig_expr_analyzer_thrifty = ThriftyDMSAnalyzer(koenig_expr_path, heavy_nt_seq=nt_wt_seqs["koenig_heavy"], light_nt_seq=nt_wt_seqs["koenig_light"])
koenig_expr_results_thrifty = koenig_expr_analyzer_thrifty.compute_results(t_values, chains=["heavy", "light"])

selection_scores_koenig = koenig_expr_analyzer_cosine.selection_score(koenig_expr_results_cosine, koenig_expr_results_thrifty)

# Adams
adams_path = EXPRESSION_DIR / "adams2017measuring_4420-fluorescein_exp_er_agg.csv"
adams_analyzer_cosine = CosineDMSAnalyzer(adams_path, cosine_model)
adams_results_cosine = adams_analyzer_cosine.compute_results(t_values, chains=["heavy"], paired=True)

adams_analyzer_thrifty = ThriftyDMSAnalyzer(adams_path, heavy_nt_seq=nt_wt_seqs["adams_heavy"])
adams_results_thrifty = adams_analyzer_thrifty.compute_results(t_values, chains=["heavy"])

selection_scores_adams = adams_analyzer_cosine.selection_score(adams_results_cosine, adams_results_thrifty)


Found 2261 heavy chain mutations and 2014 light chain mutations


CTMC light (t=0.2): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 63/63 [00:04<00:00, 13.87it/s]


Found 2261 heavy chain mutations and 2014 light chain mutations
Computing neutral probabilities for heavy chain at bl=0.2...
Loading model ThriftyHumV0.2-59
Using cached models: /scratch/users/aakarshv/venvs/peint_env/lib/python3.10/site-packages/netam/_pretrained/thrifty-0.2.0.zip
Loading multihit model ThriftyHumV0.2-59-hc-tangshm


Thrifty heavy bl=0.2: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2261/2261 [00:01<00:00, 1419.02it/s]


Computing neutral probabilities for light chain at bl=0.2...


Thrifty light bl=0.2: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2014/2014 [00:01<00:00, 1512.30it/s]


Found 2802 heavy chain mutations and 0 light chain mutations


CTMC heavy (t=0.2): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 88/88 [00:06<00:00, 13.86it/s]


Found 2802 heavy chain mutations and 0 light chain mutations
Computing neutral probabilities for heavy chain at bl=0.2...
Loading model ThriftyHumV0.2-59
Using cached models: /scratch/users/aakarshv/venvs/peint_env/lib/python3.10/site-packages/netam/_pretrained/thrifty-0.2.0.zip
Loading multihit model ThriftyHumV0.2-59-hc-tangshm


Thrifty heavy bl=0.2: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2802/2802 [00:02<00:00, 1235.18it/s]


In [21]:
koenig_df = selection_scores_koenig[1]
adams_df = selection_scores_adams[1]

df = pd.concat([
    koenig_df["heavy"], 
    koenig_df["light"], 
    adams_df["heavy"]
], axis=1)


df.columns = ["koenig heavy", "koenig light", "adams"]
df.index = ["uncorrected", "bl=0.2"]

df.style.set_caption("Expression VEP results").format("{:.3f}")

,koenig heavy,koenig light,adams
uncorrected,0.525,0.450,0.400
bl=0.2,0.613,0.508,0.464


# Binding

In [22]:
BINDING_DIR = Path("scripts/vep/data_dms/binding")
t_values = [0.2]

# Koenig (heavy and light)
koenig_bind_path = BINDING_DIR / "Koenig2017_g6_binding.csv"
koenig_bind_analyzer_cosine = CosineDMSAnalyzer(koenig_bind_path, cosine_model)
koenig_bind_results_cosine = koenig_bind_analyzer_cosine.compute_results(t_values, chains=["heavy", "light"], paired=True)

koenig_bind_analyzer_thrifty = ThriftyDMSAnalyzer(koenig_bind_path, heavy_nt_seq=nt_wt_seqs["koenig_heavy"], light_nt_seq=nt_wt_seqs["koenig_light"])
koenig_bind_results_thrifty = koenig_bind_analyzer_thrifty.compute_results(t_values, chains=["heavy", "light"])

selection_scores_koenig = koenig_bind_analyzer_cosine.selection_score(koenig_bind_results_cosine, koenig_bind_results_thrifty)

# Petersen
petersen_222_path = BINDING_DIR / "petersen_222-1c06_processed.csv"
petersen_319_path = BINDING_DIR / "petersen_319-345_processed.csv"

petersen_222_analyzer_cosine = CosineDMSAnalyzer(petersen_222_path, cosine_model)
petersen_222_results_cosine = petersen_222_analyzer_cosine.compute_results(t_values, chains=["combined"], paired=True)
petersen_222_analyzer_thrifty = ThriftyDMSAnalyzer(petersen_222_path, heavy_nt_seq=nt_wt_seqs["petersen_222_heavy"], light_nt_seq=nt_wt_seqs["petersen_222_light"])
petersen_222_results_thrifty = petersen_222_analyzer_thrifty.compute_results(t_values, chains=["combined"])
selection_scores_petersen_222 = petersen_222_analyzer_cosine.selection_score(petersen_222_results_cosine, petersen_222_results_thrifty)

petersen_319_analyzer_cosine = CosineDMSAnalyzer(petersen_319_path, cosine_model)
petersen_319_results_cosine = petersen_319_analyzer_cosine.compute_results(t_values, chains=["combined"], paired=True)
petersen_319_analyzer_thrifty = ThriftyDMSAnalyzer(petersen_319_path, heavy_nt_seq=nt_wt_seqs["petersen_319_heavy"], light_nt_seq=nt_wt_seqs["petersen_319_light"])
petersen_319_results_thrifty = petersen_319_analyzer_thrifty.compute_results(t_values, chains=["combined"])
selection_scores_petersen_319 = petersen_319_analyzer_cosine.selection_score(petersen_319_results_cosine, petersen_319_results_thrifty) 

# Shane
shane_119_path = BINDING_DIR / "shanehsazzadeh_trastuzumab_119.csv"
shane_120_path = BINDING_DIR / "shanehsazzadeh_trastuzumab_120.csv"

shane_119_analyzer_cosine = CosineDMSAnalyzer(shane_119_path, cosine_model)
shane_119_results_cosine = shane_119_analyzer_cosine.compute_results(t_values, chains=["heavy"], paired=True)
shane_119_analyzer_thrifty = ThriftyDMSAnalyzer(shane_119_path, heavy_nt_seq=nt_wt_seqs["shane_119_heavy"])
shane_119_results_thrifty = shane_119_analyzer_thrifty.compute_results(t_values, chains=["heavy"])
selection_scores_shane_119 = shane_119_analyzer_cosine.selection_score(shane_119_results_cosine, shane_119_results_thrifty)

shane_120_analyzer_cosine = CosineDMSAnalyzer(shane_120_path, cosine_model)
shane_120_results_cosine = shane_120_analyzer_cosine.compute_results(t_values, chains=["heavy"], paired=True)
shane_120_analyzer_thrifty = ThriftyDMSAnalyzer(shane_120_path, heavy_nt_seq=nt_wt_seqs["shane_120_heavy"])
shane_120_results_thrifty = shane_120_analyzer_thrifty.compute_results(t_values, chains=["heavy"])
selection_scores_shane_120 = shane_120_analyzer_cosine.selection_score(shane_120_results_cosine, shane_120_results_thrifty)

Found 2261 heavy chain mutations and 2014 light chain mutations


CTMC light (t=0.2): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 63/63 [00:04<00:00, 13.82it/s]


Found 2261 heavy chain mutations and 2014 light chain mutations
Computing neutral probabilities for heavy chain at bl=0.2...
Loading model ThriftyHumV0.2-59
Using cached models: /scratch/users/aakarshv/venvs/peint_env/lib/python3.10/site-packages/netam/_pretrained/thrifty-0.2.0.zip
Loading multihit model ThriftyHumV0.2-59-hc-tangshm


Thrifty heavy bl=0.2: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2261/2261 [00:01<00:00, 1418.81it/s]


Computing neutral probabilities for light chain at bl=0.2...


...

Thrifty heavy bl=0.2: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 184/184 [00:00<00:00, 849.04it/s]


Found 201 heavy chain mutations and 0 light chain mutations


CTMC heavy (t=0.2): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 14.14it/s]


Found 201 heavy chain mutations and 0 light chain mutations
Computing neutral probabilities for heavy chain at bl=0.2...
Loading model ThriftyHumV0.2-59
Using cached models: /scratch/users/aakarshv/venvs/peint_env/lib/python3.10/site-packages/netam/_pretrained/thrifty-0.2.0.zip
Loading multihit model ThriftyHumV0.2-59-hc-tangshm


Thrifty heavy bl=0.2: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 771.59it/s]


In [23]:
koenig_df = selection_scores_koenig[1]
petersen_222_df = selection_scores_petersen_222[1]
petersen_319_df = selection_scores_petersen_319[1]
shane_119_df = selection_scores_shane_119[1]
shane_120_df = selection_scores_shane_120[1]

df = pd.concat([
    koenig_df["heavy"], 
    koenig_df["light"], 
    shane_119_df["heavy"],
    shane_120_df["heavy"],
    petersen_222_df["combined"],
    petersen_319_df["combined"]
], axis=1)


df.columns = ["koenig heavy", "koenig light", "shane 119", "shane 120","petersen 222", "petersen 319"]
df.index = ["uncorrected", "bl=0.2"]

df.style.set_caption("Binding VEP results").format("{:.3f}")

,koenig heavy,koenig light,shane 119,shane 120,petersen 222,petersen 319
uncorrected,0.336,0.323,0.462,0.454,0.308,0.393
bl=0.2,0.456,0.371,0.498,0.536,0.328,0.504
